# RECAST Baseline Evaluation Driver

**Plug-and-play notebook for evaluating any HuggingFace model on RECAST-30K.**

## How to use
1. Fill in your **model name** (any HuggingFace model ID) and **Google Drive project path** in Section 0
2. Run All Cells
3. Results CSV appears in `<your_project_dir>/output/`

## Expected file structure
```
MyDrive/<project_dir>/
├── data/                  # Put recast_30k.json here
│   └── recast_30k.json
├── output/                # Baseline + finetuned CSVs written here
│   ├── baseline_<model>_<timestamp>.csv
│   └── baseline_<model>_<timestamp>_meta.json
├── scores/                # Cross-model comparison (from recast_eval.ipynb)
├── constraint_checker.py  # Auto-copied from this repo
├── evaluator.py           # Auto-copied from this repo
└── driver.ipynb           # This notebook
```

## Section 0: Configuration

Enter your model and project path below, then run the cell.

In [ ]:
#@title Configuration { display-mode: "form" }
#@markdown ### Required
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  #@param {type:"string"}
GDRIVE_PROJECT_DIR = "MyDrive/recast_eval"  #@param {type:"string"}

#@markdown ### Optional
HF_TOKEN = ""  #@param {type:"string"}
USE_4BIT = True  #@param {type:"boolean"}
MAX_NEW_TOKENS = 512  #@param {type:"integer"}
BATCH_SIZE = 4  #@param {type:"integer"}
NUM_SAMPLES = 0  #@param {type:"integer"}

# Derived paths
DRIVE_BASE = f"/content/drive/{GDRIVE_PROJECT_DIR}"
DATA_DIR = f"{DRIVE_BASE}/data"
OUTPUT_DIR = f"{DRIVE_BASE}/output"
DATASET_PATH = f"{DATA_DIR}/recast_30k.json"

print(f"Model:       {MODEL_NAME}")
print(f"Project dir: {DRIVE_BASE}")
print(f"Data dir:    {DATA_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Dataset:     {DATASET_PATH}")
print(f"4-bit quant: {USE_4BIT}")
print(f"Max tokens:  {MAX_NEW_TOKENS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Num samples: {NUM_SAMPLES if NUM_SAMPLES > 0 else 'all'}")

## Section 1: Setup

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes torch tqdm pandas matplotlib seaborn huggingface_hub

In [ ]:
import os
import shutil
import sys

from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Directories ready.")

# Copy helper modules to Drive so they persist across sessions
for fname in ["constraint_checker.py", "evaluator.py"]:
    src = f"/content/{fname}"
    dst = f"{DRIVE_BASE}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {fname} -> {dst}")
    elif os.path.exists(dst):
        shutil.copy2(dst, src)
        print(f"Restored {fname} from Drive")
    else:
        print(f"WARNING: {fname} not found. Upload it to /content/ or {DRIVE_BASE}/")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Verify dataset exists
assert os.path.exists(DATASET_PATH), (
    f"Dataset not found at {DATASET_PATH}. "
    f"Upload recast_30k.json to {DATA_DIR}/"
)

## Section 2: Load Model

Downloads and loads whatever HuggingFace model you specified above. Uses 4-bit quantization by default for T4 compatibility.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")

print(f"Downloading and loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

model.eval()
print(f"Model loaded: {MODEL_NAME}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Section 3: Load Dataset

In [ ]:
import json
import re
import random
from collections import Counter, defaultdict

# --- Parsing helpers ---

def infer_difficulty(num_constraints):
    if num_constraints <= 2:
        return "L1"
    elif num_constraints <= 4:
        return "L2"
    elif num_constraints <= 7:
        return "L3"
    else:
        return "L4"

def parse_record(record, idx):
    prompt = (record.get("winner_prompt")
              or record.get("prompt")
              or record.get("instruction")
              or "")

    constraints = record.get("constraints", record.get("constraint_list", None))
    if constraints is None:
        constraints = []
    if isinstance(constraints, str):
        try:
            constraints = json.loads(constraints)
        except json.JSONDecodeError:
            constraints = [{"type": "raw", "description": constraints}]
    if not isinstance(constraints, list):
        constraints = [constraints] if constraints else []

    difficulty = (record.get("difficulty_level")
                  or record.get("difficulty")
                  or record.get("level"))
    if difficulty is None:
        difficulty = infer_difficulty(len(constraints))
    difficulty = str(difficulty)
    if not difficulty.startswith("L"):
        difficulty = f"L{difficulty}"

    reference = (record.get("winner_response")
                 or record.get("response")
                 or record.get("output")
                 or "")

    return {
        "id": record.get("id", idx),
        "prompt": prompt,
        "constraints": constraints,
        "num_constraints": len(constraints),
        "difficulty_level": difficulty,
        "reference_response": reference,
    }

# --- Load ---

raw_data = []
if DATASET_PATH.endswith(".jsonl"):
    with open(DATASET_PATH, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                raw_data.append(json.loads(line))
else:
    with open(DATASET_PATH, "r") as f:
        loaded = json.load(f)
        if isinstance(loaded, list):
            raw_data = loaded
        elif isinstance(loaded, dict):
            for key in ["data", "instances", "examples", "records"]:
                if key in loaded and isinstance(loaded[key], list):
                    raw_data = loaded[key]
                    break
            if not raw_data:
                raw_data = [loaded]

dataset = [parse_record(r, i) for i, r in enumerate(raw_data)]

# Stratified sampling if requested
if NUM_SAMPLES > 0:
    by_level = defaultdict(list)
    for item in dataset:
        by_level[item["difficulty_level"]].append(item)
    per_level = max(1, NUM_SAMPLES // len(by_level))
    sampled = []
    for level in sorted(by_level.keys()):
        items = by_level[level]
        random.seed(42)
        sampled.extend(random.sample(items, min(per_level, len(items))))
    dataset = sampled
    print(f"Stratified sampled to {len(dataset)} instances")

level_counts = Counter(d["difficulty_level"] for d in dataset)
print(f"Total instances: {len(dataset)}")
for level in sorted(level_counts.keys()):
    print(f"  {level}: {level_counts[level]}")

## Section 4: Inference

Runs the model on all prompts. Checkpoints to Drive every 50 batches.

In [ ]:
import time
from tqdm.auto import tqdm

MODEL_NAME_SAFE = MODEL_NAME.replace("/", "_").replace("-", "_")
checkpoint_path = os.path.join(OUTPUT_DIR, f"baseline_{MODEL_NAME_SAFE}_responses.jsonl")

# Resume from checkpoint
completed_ids = set()
inference_results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                completed_ids.add(rec["id"])
                inference_results.append(rec)
    print(f"Resumed: {len(completed_ids)} already completed.")

remaining = [d for d in dataset if d["id"] not in completed_ids]
print(f"Remaining: {len(remaining)}")

has_chat_template = hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None

def format_prompt(prompt_text):
    if has_chat_template:
        messages = [{"role": "user", "content": prompt_text}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"User: {prompt_text}\nAssistant:"

# --- Run inference ---
inference_start = time.time()
batch_count = 0
checkpoint_file = open(checkpoint_path, "a")

try:
    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Inference"):
        batch = remaining[i : i + BATCH_SIZE]
        prompts = [format_prompt(item["prompt"]) for item in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        for j, item in enumerate(batch):
            generated_tokens = outputs[j][input_len:]
            response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

            result = {
                "id": item["id"],
                "prompt": item["prompt"],
                "response": response,
                "constraints": item["constraints"],
                "num_constraints": item["num_constraints"],
                "difficulty_level": item["difficulty_level"],
            }
            inference_results.append(result)
            checkpoint_file.write(json.dumps(result) + "\n")

        batch_count += 1
        if batch_count % 50 == 0:
            checkpoint_file.flush()
            print(f"  Checkpoint at batch {batch_count} ({len(inference_results)} total)")
        if batch_count % 100 == 0:
            torch.cuda.empty_cache()
finally:
    checkpoint_file.close()

inference_time = time.time() - inference_start
print(f"\nInference complete: {len(inference_results)} results in {inference_time:.1f}s")
torch.cuda.empty_cache()

## Section 5: Evaluate & Save Results

Runs deterministic constraint checking and saves the scored CSV to `output/`.

In [ ]:
from evaluator import evaluate_responses, compute_metrics, save_results_csv, print_summary

eval_start = time.time()
evaluate_responses(inference_results)
metrics = compute_metrics(inference_results)
eval_time = time.time() - eval_start

total_time = inference_time + eval_time

csv_path = save_results_csv(
    results=inference_results,
    metrics=metrics,
    output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME,
    label="baseline",
    elapsed_seconds=total_time,
)

print_summary(metrics, MODEL_NAME, label="baseline", elapsed_seconds=total_time)
print(f"Results saved to: {csv_path}")

## Section 6: Visualization

In [ ]:
from viz_utils import plot_csr_degradation, plot_per_type_bar, plot_constraint_distribution

plot_csr_degradation(metrics["by_level"], MODEL_NAME, OUTPUT_DIR)
plot_per_type_bar(metrics["per_type"], MODEL_NAME, OUTPUT_DIR)

dist_data = [
    {"num_constraints": r["num_constraints"], "difficulty_level": r["difficulty_level"]}
    for r in inference_results
]
plot_constraint_distribution(dist_data, MODEL_NAME, OUTPUT_DIR)
print(f"Plots saved to {OUTPUT_DIR}")

---

## (Future) Finetuned Model Evaluation

To evaluate a finetuned model later, load it the same way and call the same
evaluation functions with `label="finetuned"`. The evaluator module provides
the same `evaluate_responses()`, `compute_metrics()`, and `save_results_csv()`
functions — just pass `label="finetuned"` to get properly labeled output files.

Example:
```python
# After running inference with a finetuned model into finetuned_results:
from evaluator import evaluate_responses, compute_metrics, save_results_csv, print_summary

evaluate_responses(finetuned_results)
metrics = compute_metrics(finetuned_results)
csv_path = save_results_csv(
    results=finetuned_results,
    metrics=metrics,
    output_dir=OUTPUT_DIR,
    model_name="my-finetuned-model",
    label="finetuned",
    elapsed_seconds=total_time,
)
print_summary(metrics, "my-finetuned-model", label="finetuned", elapsed_seconds=total_time)
```